# 초기 100사이클 Feature Engineering

cycle 2~100의 요약 신호와 cycle 10/100의 ΔQ(V)를 사용해 셀 단위 모델 입력표를 만든다. 후기 수명, knee point 등 미래 정보는 모델 입력에서 제외한다.

In [1]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
sys.path.insert(0, str(PROJECT_ROOT))
from src.features import FEATURE_SETS, extract_feature_table

DATA_DIR = PROJECT_ROOT / 'data'
FEATURE_PATH = PROJECT_ROOT / 'results' / 'feature_table.csv'
pd.set_option('display.max_columns', 100)

## 피처 테이블 생성

원본 MAT 파일에서 다시 추출하려면 `REBUILD_FEATURES = True`로 변경한다. 파일이 이미 있으면 빠르게 기존 결과를 읽는다.

In [2]:
REBUILD_FEATURES = False
if REBUILD_FEATURES or not FEATURE_PATH.exists():
    feature_df = extract_feature_table(DATA_DIR, FEATURE_PATH)
else:
    feature_df = pd.read_csv(FEATURE_PATH)

print('shape:', feature_df.shape)
display(feature_df.head())

shape: (139, 19)


,cell_key,cell_index,batch,batch_date,cycle_life,policy_readable,first_stage_c_rate,max_c_rate,initial_QDischarge,QDischarge_mean_100,QDischarge_cycle100_minus_cycle10,IR_mean_100,temperature_slope_100,chargetime_mean_100,dq_mean,dq_std,dq_min,dq_range,dq_kurtosis
0,2017-05-12_cell_1,0,Batch 1,2017-05-12,1190.0,3.6C(80%)-3.6C,3.6,3.6,1.070689,1.080597,0.001375,0.016643,-0.001193,13.384231,-0.002873,0.003110,-0.008460,0.009228,-1.349054
1,2017-05-12_cell_2,1,Batch 1,2017-05-12,1179.0,3.6C(80%)-3.6C,3.6,3.6,1.075301,1.081247,0.000853,0.016922,-0.003096,13.374696,-0.004100,0.003113,-0.011004,0.010916,-1.028256
2,2017-05-12_cell_3,2,Batch 1,2017-05-12,1177.0,3.6C(80%)-3.6C,3.6,3.6,1.079922,1.085371,0.001159,0.016769,-0.003094,13.378886,-0.004487,0.004283,-0.017216,0.017229,0.356410
3,2017-05-12_cell_4,3,Batch 1,2017-05-12,1226.0,4C(80%)-4C,4.0,4.0,1.079723,1.084949,0.000442,0.016311,-0.002254,12.053277,-0.007456,0.006010,-0.018961,0.019143,-1.094889
4,2017-05-12_cell_5,4,Batch 1,2017-05-12,1227.0,4C(80%)-4C,4.0,4.0,1.078417,1.082844,0.000692,0.016669,-0.000487,12.053314,-0.005750,0.004746,-0.013958,0.014004,-1.334500


## 품질 및 누수 점검

배치별 표본 수, 결측치, 최종 후보 피처를 확인한다.

In [3]:
display(feature_df.groupby('batch').agg(cells=('cell_key', 'size'), labeled=('cycle_life', 'count')))
display(feature_df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('모델 후보 피처 세트:')
for name, columns in FEATURE_SETS.items():
    print(f'- {name}: {columns}')

,cells,labeled
batch,,
Batch 1,46,46
Batch 2,47,39
Batch 3,46,44


,missing
cycle_life,10
IR_mean_100,7
cell_key,0
QDischarge_cycle100_minus_cycle10,0
dq_range,0
dq_min,0
dq_std,0
dq_mean,0
chargetime_mean_100,0
temperature_slope_100,0


모델 후보 피처 세트:
- base_initial_q: ['dq_min', 'dq_kurtosis', 'initial_QDischarge', 'IR_mean_100', 'temperature_slope_100', 'chargetime_mean_100', 'max_c_rate']
- base_mean_q: ['dq_min', 'dq_kurtosis', 'QDischarge_mean_100', 'IR_mean_100', 'temperature_slope_100', 'chargetime_mean_100', 'max_c_rate']
- no_dq_shape: ['dq_min', 'initial_QDischarge', 'IR_mean_100', 'temperature_slope_100', 'chargetime_mean_100', 'max_c_rate']
- dq_std_substitution: ['dq_std', 'dq_kurtosis', 'initial_QDischarge', 'IR_mean_100', 'temperature_slope_100', 'chargetime_mean_100', 'max_c_rate']
- dq_mean_substitution: ['dq_mean', 'dq_kurtosis', 'initial_QDischarge', 'IR_mean_100', 'temperature_slope_100', 'chargetime_mean_100', 'max_c_rate']
- dq_range_substitution: ['dq_range', 'dq_kurtosis', 'initial_QDischarge', 'IR_mean_100', 'temperature_slope_100', 'chargetime_mean_100', 'max_c_rate']
- dq_only: ['dq_min', 'dq_kurtosis']
- no_protocol: ['dq_min', 'dq_kurtosis', 'initial_QDischarge', 'IR_mean_100', 'temperature_